# Notas de Estadística: Métodos de Remuestreo

## 1. Muestreo Aleatorio y Generación de Nuevas Muestras

Sea $X_1, X_2, \ldots, X_n$ una muestra aleatoria de una población con distribución desconocida.

**Objetivo**: Entender mejor el comportamiento de un estimador $\hat{\theta} = f(X_1, X_2, \ldots, X_n)$ sin conocer la distribución poblacional.

### Idea Central
Generar nuevas muestras a partir de la muestra original para aproximar la distribución muestral de $\hat{\theta}$.

$$
\hat{\theta} = f(X_1, X_2, \ldots, X_n) \quad \longrightarrow \quad \text{¿Cuál es } E[\hat{\theta}] \text{ y } \text{Var}(\hat{\theta}) \text{?}
$$

---

## 2. Estimadores y Sesgo (Bias)

### Definición del Sesgo

El sesgo de un estimador $\hat{\theta}$ para un parámetro $\theta$ se define como:

$$
\text{bias}(\hat{\theta}) = E[\hat{\theta}] - \theta
$$

- Si $\text{bias}(\hat{\theta}) = 0$, el estimador es **insesgado**.
- En la práctica, $E[\hat{\theta}]$ es desconocida y debe estimarse.

### Estimación del Sesgo mediante Remuestreo

Dado que no podemos repetir el muestreo de la población, usamos técnicas de **remuestreo** para aproximar $E[\hat{\theta}]$.

---

## 3. Método Jackknife

### Idea Fundamental

El método Jackknife ("navaja") estima el sesgo y la varianza de $\hat{\theta}$ eliminando sistemáticamente una observación a la vez.

### Procedimiento

1. Para cada $i = 1, 2, \ldots, n$, calcular el estimador omitiendo la $i$-ésima observación:
   $$
   \hat{\theta}_{(i)} = f(X_1, \ldots, X_{i-1}, X_{i+1}, \ldots, X_n)
   $$

2. Calcular la media de los estimadores Jackknife:
   $$
   \bar{\theta}_{(\cdot)} = \frac{1}{n} \sum_{i=1}^{n} \hat{\theta}_{(i)}
   $$

3. **Estimador Jackknife corregido para el sesgo**:
   $$
   \hat{\theta}_{\text{Jack}} = \hat{\theta} - (n-1)\left(\bar{\theta}_{(\cdot)} - \hat{\theta}\right)
   $$
   Equivalentemente:
   $$
   \hat{\theta}_{\text{Jack}} = n\hat{\theta} - (n-1)\bar{\theta}_{(\cdot)}
   $$

### Estimación de la Varianza Jackknife

$$
\widehat{\text{Var}}_{\text{Jack}}(\hat{\theta}) = \frac{n-1}{n} \sum_{i=1}^{n} \left(\hat{\theta}_{(i)} - \bar{\theta}_{(\cdot)}\right)^2
$$

### Ejemplo: Jackknife para la Media

Si $\hat{\theta} = \bar{X} = \frac{1}{n}\sum_{i=1}^{n} X_i$, entonces:

$$
\hat{\theta}_{(i)} = \frac{1}{n-1}\sum_{j \neq i} X_j = \frac{n\bar{X} - X_i}{n-1}
$$

En este caso particular, el estimador Jackknife coincide con la media original, ya que la media muestral es insesgada.

---

## 4. Método Bootstrapping

### Idea Fundamental

El Bootstrapping ("remuestreo con reemplazo") genera réplicas de la muestra original para aproximar la distribución muestral de $\hat{\theta}$.

### Procedimiento

1. Partir de una muestra aleatoria original: $X_1, X_2, \ldots, X_n$.

2. Generar una **muestra bootstrap** $X_1^*, X_2^*, \ldots, X_n^*$ muestreando **con reemplazo** de la muestra original.

3. Calcular el estimador para la muestra bootstrap:
   $$
   \hat{\theta}^* = f(X_1^*, X_2^*, \ldots, X_n^*)
   $$

4. Repetir los pasos 2-3 un total de $B$ veces (típicamente $B \geq 1000$) para obtener:
   $$
   \hat{\theta}^{*(1)}, \hat{\theta}^{*(2)}, \ldots, \hat{\theta}^{*(B)}
   $$

### Estimaciones Bootstrap

- **Media bootstrap** (aproximación de $E[\hat{\theta}]$):
  $$
  \bar{\theta}^* = \frac{1}{B} \sum_{b=1}^{B} \hat{\theta}^{*(b)}
  $$

- **Sesgo estimado**:
  $$
  \widehat{\text{bias}}_{\text{Boot}} = \bar{\theta}^* - \hat{\theta}
  $$

- **Varianza estimada**:
  $$
  \widehat{\text{Var}}_{\text{Boot}}(\hat{\theta}) = \frac{1}{B-1} \sum_{b=1}^{B} \left(\hat{\theta}^{*(b)} - \bar{\theta}^*\right)^2
  $$

- **Intervalos de confianza**: Se pueden construir usando percentiles de la distribución empírica de $\{\hat{\theta}^{*(b)}\}$.


### Diagrama Conceptual del Bootstrapping

```
Muestra Original:  [X_1, X_2, ..., X_n]
                        |
                        v
            Muestreo con reemplazo (n veces)
                        |
                        v
Muestra Bootstrap:  [X_1*, X_2*, ..., X_n*]  -->  Calcular theta*
                        |
                        v
            Repetir B veces
                        |
                        v
Distribucion Bootstrap de theta*:  {theta*^(1), theta*^(2), ..., theta*^(B)}
```

## 5. Comparación Jackknife vs. Bootstrapping

| Característica | Jackknife | Bootstrapping |
|---------------|-----------|---------------|
| **Mecanismo** | Elimina 1 observación por vez | Remuestreo con reemplazo |
| **Número de réplicas** | Exactamente $n$ | Arbitrario $B$ (típicamente $B \gg n$) |
| **Costo computacional** | Bajo ($n$ evaluaciones) | Alto ($B$ evaluaciones) |
| **Aplicabilidad** | Estimadores suaves | Casi cualquier estimador |
| **Estimación de varianza** | Fórmula cerrada | Empírica vía percentiles |
| **Sensibilidad a outliers** | Alta (cada omisión tiene peso $1/n$) | Moderada (diluido por $B$ réplicas) |

## 6. Notas Prácticas de Implementación

### Para Jackknife:
```python
# Pseudocódigo conceptual
theta_hat = f(X)  # Estimador original
theta_jack = []
for i in range(n):
    X_leave_one = X.copy()
    X_leave_one.pop(i)  # Eliminar i-ésima observación
    theta_i = f(X_leave_one)
    theta_jack.append(theta_i)
theta_bar = mean(theta_jack)
theta_jack_corrected = n*theta_hat - (n-1)*theta_bar
```

### Para Bootstrapping:
```python
# Pseudocódigo conceptual
theta_boot = []
for b in range(B):
    X_star = sample_with_replacement(X, size=n)
    theta_star = f(X_star)
    theta_boot.append(theta_star)
bias_boot = mean(theta_boot) - theta_hat
var_boot = variance(theta_boot)
```

---

## 7. Advertencias y Consideraciones

1. **Dependencia de los datos**: Ambos métodos asumen que la muestra original es representativa de la población. Si la muestra está sesgada, los resultados del remuestreo también lo estarán.

2. **Independencia**: Las observaciones $X_1, \ldots, X_n$ deben ser independientes e idénticamente distribuidas (i.i.d.). Para datos dependientes (series de tiempo, datos espaciales), se requieren variantes especializadas (block bootstrap, etc.).

3. **Estimadores no suaves**: El Jackknife puede fallar para estimadores no diferenciables (ej: mediana). El Bootstrap es más robusto en estos casos.

4. **Tamaño de muestra**: Para $n$ pequeño, el Jackknife puede ser inestable. El Bootstrap requiere $B$ grande para estabilizar las estimaciones.

---

## 8. Referencias Sugeridas

- Efron, B. (1982). *The Jackknife, the Bootstrap and Other Resampling Plans*. SIAM.
- Efron, B., & Tibshirani, R. J. (1993). *An Introduction to the Bootstrap*. Chapman & Hall/CRC.
- Davison, A. C., & Hinkley, D. V. (1997). *Bootstrap Methods and Their Application*. Cambridge University Press.